# Prepare Amazon Reviews 2023 — merge all categories (InteRecAgent preprocess)

This notebook runs on **Kaggle**. Goal: download all 34 categories of the [Amazon Reviews 2023](https://amazon-reviews-2023.github.io/) dataset (McAuley Lab), **trim fields** to keep it light, and merge everything into 2 Parquet files:

- `reviews_all.parquet`
- `meta_all.parquet`

These two files are used as input for the original `prepare_amazon.ipynb` step from InteRecAgent (replacing `All_Amazon_Meta.json.gz` / `reviews_{cat}.json.gz` from the 2014 version).

## Why trim fields?
The full version (keeping `images`, `videos`, `details`, `features`, review `text`...) across all 34 categories combined is **700GB+**. This notebook **only keeps the fields the InteRecAgent pipeline actually uses**, so the estimated total size drops to a few dozen GB — small enough to run on Kaggle.

## Differences from the 2014 version
- No more ready-made `brand` column → taken from `details["Brand"]` (not every item has it).
- The join key between review ↔ meta is `parent_asin` (not `asin`).
- Review `timestamp` is in **milliseconds** (13 digits) → the notebook converts it to seconds to stay compatible with old code that uses `unixReviewTime`.
- Meta only downloads the **raw** part (`meta_{category}.jsonl.gz`), not full review text (only keeps rating/timestamp/id).

## How to use
1. Adjust `CATEGORIES` / `DEBUG_LIMIT_ROWS` in the Config cell below if needed (comment out a line in `CATEGORIES` to skip that category).
2. Run the cells in order. The notebook supports **resume**: if it gets interrupted (Kaggle session timeout, network timeout...), rerunning will automatically skip categories that are already done.
3. The last cell merges all part-files by category into a single file for reviews and a single file for meta.

In [ ]:
%pip install -q pyarrow requests

In [ ]:
import os, gzip, json, time, shutil, logging
import requests
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd

## Config

In [ ]:
# All 34 official categories (33 domains + Unknown) per amazon-reviews-2023.github.io
# Comment out a line to skip that category
CATEGORIES = [
    "All_Beauty",
    "Amazon_Fashion",
    "Appliances",
    "Arts_Crafts_and_Sewing",
    "Automotive",
    "Baby_Products",
    "Beauty_and_Personal_Care",
    "Books",
    "CDs_and_Vinyl",
    "Cell_Phones_and_Accessories",
    "Clothing_Shoes_and_Jewelry",
    "Digital_Music",
    "Electronics",
    "Gift_Cards",
    "Grocery_and_Gourmet_Food",
    "Handmade_Products",
    "Health_and_Household",
    "Health_and_Personal_Care",
    "Home_and_Kitchen",
    "Industrial_and_Scientific",
    "Kindle_Store",
    "Magazine_Subscriptions",
    "Movies_and_TV",
    "Musical_Instruments",
    "Office_Products",
    "Patio_Lawn_and_Garden",
    "Pet_Supplies",
    "Software",
    "Sports_and_Outdoors",
    "Subscription_Boxes",
    "Tools_and_Home_Improvement",
    "Toys_and_Games",
    "Video_Games",
    # "Unknown",
]

In [ ]:
# URL config - DO NOT TOUCH
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
REVIEW_URL_TMPL = f"{BASE_URL}/review_categories/{{cat}}.jsonl.gz"
META_URL_TMPL = f"{BASE_URL}/meta_categories/meta_{{cat}}.jsonl.gz"

In [ ]:
# Set > 0 to test quickly (limits rows/category); set None to run in full
DEBUG_LIMIT_ROWS = None  # e.g. 5000 to test the pipeline before running full

BATCH_SIZE = 50_000        # number of rows accumulated before flushing to parquet
MAX_RETRIES = 5
RETRY_SLEEP_SEC = 10
LOG_EVERY_N_ROWS = 200_000

In [ ]:
# Output path config - MUST CHECK BEOFRE RUN LOCAL
OUTPUT_DIR = "/kaggle/working/amazon2023"
REVIEW_PARTS_DIR = os.path.join(OUTPUT_DIR, "reviews_by_category")
META_PARTS_DIR = os.path.join(OUTPUT_DIR, "meta_by_category")
os.makedirs(REVIEW_PARTS_DIR, exist_ok=True)
os.makedirs(META_PARTS_DIR, exist_ok=True)

REVIEW_ALL_PATH = os.path.join(OUTPUT_DIR, "reviews_all.parquet")
META_ALL_PATH = os.path.join(OUTPUT_DIR, "meta_all.parquet")

LOG_DIR = os.path.join(OUTPUT_DIR, "logs")
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = os.path.join(LOG_DIR, time.strftime("prepare_amazon_2023_%Y%m%d_%H%M%S.log"))

## Logging

In [ ]:
logger = logging.getLogger("prepare_amazon_2023")
logger.setLevel(logging.INFO)
logger.handlers.clear()  # avoid duplicate handlers if this cell is re-run

formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")

console_handler = logging.StreamHandler()
console_handler.setFormatter(formatter)
logger.addHandler(console_handler)

file_handler = logging.FileHandler(LOG_FILE)
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

logger.info(f"Logging to {LOG_FILE}")

## Schema (comment out a field to drop it)

Every field the raw dataset provides is listed below, one per line, same style as `CATEGORIES`. To exclude a field, just comment out its line — `review_record_to_row` / `meta_record_to_row` build each row from `REVIEW_SCHEMA.names` / `META_SCHEMA.names`, so nothing else needs to change.

Nested fields (`images`, `videos`, `details`, `features`, `bought_together`) are stored as JSON-encoded strings so they can stay simple one-line entries too. `category` is kept as a native list of strings since it's already flat.

In [ ]:
REVIEW_SCHEMA = pa.schema([
    ("user_id", pa.string()),
    ("item_id", pa.string()),           # = parent_asin
    # ("asin", pa.string()),
    ("rating", pa.float32()),
    # ("title", pa.string()),             # review title
    # ("text", pa.string()),              # review body
    ("timestamp", pa.int64()),          # seconds (converted from milliseconds if needed)
    # ("helpful_vote", pa.int64()),
    # ("verified_purchase", pa.bool_()),
    # ("images", pa.string()),            # JSON-encoded list of image dicts
    ("category_domain", pa.string()),   # source category file, e.g. "Beauty_and_Personal_Care"
])

In [ ]:
META_SCHEMA = pa.schema([
    ("item_id", pa.string()),           # = parent_asin
    # ("main_category", pa.string()),
    ("title", pa.string()),
    # ("average_rating", pa.float32()),
    # ("rating_number", pa.int64()),
    # ("features", pa.string()),          # JSON-encoded list of bullet points
    ("description", pa.string()),       # original description list joined with \n\n
    ("price", pa.float32()),
    # ("images", pa.string()),            # JSON-encoded list of image dicts
    # ("videos", pa.string()),            # JSON-encoded list of video dicts
    # ("store", pa.string()),
    ("category", pa.list_(pa.string())),# original hierarchical categories
    # ("details", pa.string()),           # JSON-encoded dict of item details
    ("brand", pa.string()),             # taken from details['Brand'] if present
    # ("bought_together", pa.string()),   # JSON-encoded, usually null
    ("category_domain", pa.string()),   # source category file
])

## Helper functions

In [ ]:
def iter_jsonl_gz_stream(url, max_retries=MAX_RETRIES):
    """Download and decompress .jsonl.gz on-the-fly (WITHOUT saving the .gz file to disk), yield each dict."""
    attempt = 0
    while True:
        attempt += 1
        try:
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                with gzip.GzipFile(fileobj=r.raw) as gz:
                    for raw_line in gz:
                        line = raw_line.strip()
                        if not line:
                            continue
                        try:
                            yield json.loads(line)
                        except json.JSONDecodeError:
                            continue
            return  # stream completed successfully
        except (requests.exceptions.RequestException, gzip.BadGzipFile, EOFError) as e:
            if attempt >= max_retries:
                raise
            print(f"  [retry {attempt}/{max_retries}] Error downloading {url}: {e}. Retrying in {RETRY_SLEEP_SEC}s...")
            time.sleep(RETRY_SLEEP_SEC)

In [ ]:
def parse_price(raw_price):
    if raw_price is None:
        return None
    if isinstance(raw_price, (int, float)):
        return float(raw_price)
    s = str(raw_price).strip()
    if s == "" or s.lower() == "none":
        return None
    s = s.replace("$", "").replace(",", "")
    try:
        return float(s)
    except ValueError:
        return None

In [ ]:
def extract_brand(details):
    if not isinstance(details, dict):
        return None
    for key in details:
        if key.strip().lower() == "brand":
            return details[key]
    return None

In [ ]:
def normalize_timestamp(ts):
    if ts is None:
        return None
    try:
        ts = int(ts)
    except (ValueError, TypeError):
        return None
    # original timestamp is in milliseconds (13 digits) -> convert to seconds for compatibility with old code
    if ts > 10**12:
        ts = ts // 1000
    return ts

In [ ]:
def to_json_or_none(value):
    """Encode a nested list/dict field to a JSON string so it can live in a plain pa.string() column."""
    if value is None:
        return None
    return json.dumps(value, ensure_ascii=False)

In [ ]:
def review_record_to_row(rec, category_domain):
    # Build every possible field first, then keep only the ones still present in REVIEW_SCHEMA
    # (i.e. not commented out above). This way, toggling a field only requires editing the schema.
    full_row = {
        "user_id": rec.get("user_id"),
        "item_id": rec.get("parent_asin") or rec.get("asin"),
        "asin": rec.get("asin"),
        "rating": rec.get("rating"),
        "title": rec.get("title"),
        "text": rec.get("text"),
        "timestamp": normalize_timestamp(rec.get("timestamp")),
        "helpful_vote": rec.get("helpful_vote"),
        "verified_purchase": rec.get("verified_purchase"),
        "images": to_json_or_none(rec.get("images")),
        "category_domain": category_domain,
    }
    return {name: full_row.get(name) for name in REVIEW_SCHEMA.names}

In [ ]:
def meta_record_to_row(rec, category_domain):
    desc = rec.get("description") or []
    if isinstance(desc, list):
        desc = "\n\n".join([d for d in desc if isinstance(d, str) and d.strip()]) or None

    cats = rec.get("categories") or []
    if not isinstance(cats, list):
        cats = []

    # Same pattern as review_record_to_row: build every possible field, then filter down
    # to whatever fields are still present in META_SCHEMA.
    full_row = {
        "item_id": rec.get("parent_asin"),
        "main_category": rec.get("main_category"),
        "title": rec.get("title"),
        "average_rating": rec.get("average_rating"),
        "rating_number": rec.get("rating_number"),
        "features": to_json_or_none(rec.get("features")),
        "description": desc,
        "price": parse_price(rec.get("price")),
        "images": to_json_or_none(rec.get("images")),
        "videos": to_json_or_none(rec.get("videos")),
        "store": rec.get("store"),
        "category": cats,
        "details": to_json_or_none(rec.get("details")),
        "brand": extract_brand(rec.get("details")),
        "bought_together": to_json_or_none(rec.get("bought_together")),
        "category_domain": category_domain,
    }
    return {name: full_row.get(name) for name in META_SCHEMA.names}

In [ ]:
# Process 1 category -> write 1 part-file parquet (with resume)
def process_category(cat, part, url_tmpl, out_dir, schema, record_to_row_fn):
    """part: 'reviews' or 'meta'. Returns number of rows written."""
    out_path = os.path.join(out_dir, f"{cat}.parquet")
    done_marker = out_path + ".done"

    if os.path.exists(done_marker):
        logger.info(f"[{part}] {cat}: already done, skipping (resume).")
        return 0

    # If an incomplete old file exists (interrupted midway), delete it to rewrite from scratch
    if os.path.exists(out_path):
        os.remove(out_path)

    url = url_tmpl.format(cat=cat)
    writer = pq.ParquetWriter(out_path, schema, compression="zstd")
    buffer = []
    total_rows = 0
    next_log_at = LOG_EVERY_N_ROWS
    start_time = time.time()

    logger.info(f"[{part}] {cat}: starting download from {url}")

    try:
        for rec in iter_jsonl_gz_stream(url):
            row = record_to_row_fn(rec, cat)
            buffer.append(row)
            total_rows += 1

            if len(buffer) >= BATCH_SIZE:
                table = pa.Table.from_pylist(buffer, schema=schema)
                writer.write_table(table)
                buffer = []

            if total_rows >= next_log_at:
                elapsed = time.time() - start_time
                rate = total_rows / elapsed if elapsed > 0 else 0
                logger.info(f"[{part}] {cat}: {total_rows:,} rows so far ({rate:,.0f} rows/s)")
                next_log_at += LOG_EVERY_N_ROWS

            if DEBUG_LIMIT_ROWS is not None and total_rows >= DEBUG_LIMIT_ROWS:
                break

        if buffer:
            table = pa.Table.from_pylist(buffer, schema=schema)
            writer.write_table(table)
    finally:
        writer.close()

    with open(done_marker, "w") as f:
        f.write(str(total_rows))

    elapsed = time.time() - start_time
    logger.info(f"[{part}] {cat}: wrote {total_rows:,} rows in {elapsed:.1f}s -> {out_path}")
    return total_rows

In [ ]:
# Process all categories
def process_all_categories(categories):
    """Run reviews + meta extraction for every category, printing progress and disk usage. Returns the summary dict."""
    print(f"Will process {len(categories)} categories: {categories}")

    summary = {"reviews": {}, "meta": {}}

    for cat in categories:
        print(f"\n===== {cat} =====")
        n = process_category(cat, "reviews", REVIEW_URL_TMPL, REVIEW_PARTS_DIR, REVIEW_SCHEMA, review_record_to_row)
        summary["reviews"][cat] = n
        n = process_category(cat, "meta", META_URL_TMPL, META_PARTS_DIR, META_SCHEMA, meta_record_to_row)
        summary["meta"][cat] = n

        # log disk usage to keep track of Kaggle disk space
        total_bytes = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, fnames in os.walk(OUTPUT_DIR) for f in fnames
        )
        if total_bytes < 1e9:
            logger.info(f"Current output size: {total_bytes / 1e6:.2f} MB")
        else:
            logger.info(f"Current output size: {total_bytes / 1e9:.2f} GB")

    print("\n=== SUMMARY ===")
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    return summary

In [ ]:
# Merge all part-files (by category) into a single file
# Uses row-group based reading/writing (doesn't load everything into RAM), so it can merge even when the total data is larger than RAM.
def merge_parquet_parts(parts_dir, output_path, schema):
    part_files = sorted(
        os.path.join(parts_dir, f) for f in os.listdir(parts_dir) if f.endswith(".parquet")
    )
    if not part_files:
        logger.info(f"No files found in {parts_dir}, skipping.")
        return

    if os.path.exists(output_path):
        os.remove(output_path)

    writer = pq.ParquetWriter(output_path, schema, compression="zstd")
    total = 0
    start_time = time.time()
    try:
        for i, fp in enumerate(part_files, start=1):
            pf = pq.ParquetFile(fp)
            for rg_idx in range(pf.num_row_groups):
                table = pf.read_row_group(rg_idx)
                writer.write_table(table)
                total += table.num_rows
            logger.info(f"Merged {i}/{len(part_files)} files -> {os.path.basename(output_path)} ({total:,} rows so far)")
    finally:
        writer.close()
    elapsed = time.time() - start_time
    logger.info(f"Merged {len(part_files)} files -> {output_path} ({total:,} rows) in {elapsed:.1f}s")

In [ ]:
# Dedup meta by `item_id`
'''
Since Amazon assigns a single product to multiple categories at once, 
`meta_all.parquet` may contain items duplicated across category files. 
The cell below scans by row-group (streaming) and keeps only the first occurrence of each `item_id`, 
matching the behavior of `drop_duplicates(subset=['item_id'], keep='first')` in the original `prepare_amazon.ipynb`.
'''
def dedup_meta_by_item_id(input_path, output_path, schema):
    if not os.path.exists(input_path):
        logger.info(f"{input_path} not found, skipping dedup.")
        return

    seen_ids = set()
    tmp_path = output_path + ".tmp"
    writer = pq.ParquetWriter(tmp_path, schema, compression="zstd")
    total_in, total_out = 0, 0
    log_every_n_row_groups = 20
    start_time = time.time()
    try:
        pf = pq.ParquetFile(input_path)
        n_row_groups = pf.num_row_groups
        for rg_idx in range(n_row_groups):
            table = pf.read_row_group(rg_idx)
            rows = table.to_pylist()
            total_in += len(rows)
            keep_rows = []
            for row in rows:
                iid = row["item_id"]
                if iid is None or iid in seen_ids:
                    continue
                seen_ids.add(iid)
                keep_rows.append(row)
            if keep_rows:
                writer.write_table(pa.Table.from_pylist(keep_rows, schema=schema))
                total_out += len(keep_rows)

            if (rg_idx + 1) % log_every_n_row_groups == 0 or (rg_idx + 1) == n_row_groups:
                logger.info(f"Dedup meta: row group {rg_idx + 1}/{n_row_groups}, {total_in:,} -> {total_out:,} rows so far")
    finally:
        writer.close()

    shutil.move(tmp_path, output_path)
    elapsed = time.time() - start_time
    logger.info(f"Dedup meta: {total_in:,} -> {total_out:,} rows (kept first occurrence) in {elapsed:.1f}s.")

## Run for all categories and merge files

In [ ]:
summary = process_all_categories(CATEGORIES)
merge_parquet_parts(REVIEW_PARTS_DIR, REVIEW_ALL_PATH, REVIEW_SCHEMA)
merge_parquet_parts(META_PARTS_DIR, META_ALL_PATH, META_SCHEMA)
dedup_meta_by_item_id(META_ALL_PATH, META_ALL_PATH, META_SCHEMA)

## Quick sanity check

In [ ]:
if os.path.exists(REVIEW_ALL_PATH):
    df_r = pd.read_parquet(REVIEW_ALL_PATH)
    print("reviews_all:", df_r.shape)
    display(df_r.sample(min(5, len(df_r))))

if os.path.exists(META_ALL_PATH):
    df_m = pd.read_parquet(META_ALL_PATH)
    print("meta_all:", df_m.shape)
    display(df_m.sample(min(5, len(df_m))))

## Continue with `prepare_amazon.ipynb` (InteRecAgent)

In the original `prepare_amazon.ipynb` notebook, replace the part that reads `reviews.tsv` / `meta.tsv` with:

```python
review_df = pd.read_parquet("/kaggle/working/amazon2023/reviews_all.parquet")
meta_df = pd.read_parquet("/kaggle/working/amazon2023/meta_all.parquet")

# If you only want to work with one specific domain (similar to the old DATASET_NAME='Beauty' logic):
# review_df = review_df[review_df['category_domain'] == 'Beauty_and_Personal_Care']
# meta_df = meta_df[meta_df['category_domain'] == 'Beauty_and_Personal_Care']
```

Columns already match the names used by the old pipeline: `user_id, item_id, rating, timestamp` (review) and `item_id, title, category, price, description, brand` (meta) — the downstream `keepFirstFilter`, `lowRatingFilter`, `kCoreFilter`, `map_id`, train/valid/test split steps work as-is without modification.